# CIRA — build the pixel-preserving food-scene dataset on Google Colab

This notebook runs `data_pipeline/` end to end: **cutouts → backgrounds → splits → groups → questions → renders → audits → freeze**.

**Runtime:** *Runtime → Change runtime type → GPU* (T4 works; L4/A100 is faster). Turn on *High-RAM* if available.

**Rough timings on a T4:** Food-101 download 5–10 min · cutouts 1–2 h · Qwen check ~15 min · backgrounds fetch 40–90 min (Wikimedia rate limit) · renders 20–40 min · full audit 20–40 min.

**Disconnects:** data is built on fast local disk (`/content/cira_data`) and saved to Google Drive with `save_checkpoint()` after each step. After a disconnect, re-run the *Setup* cells; they restore the latest checkpoint. Cutout extraction resumes where it stopped.

> ⚠️ **Open items** (class list, slot positions, Wikimedia categories, benchmark name) are still proposals in `config.yaml` / `classes.py`. Run Steps 2–3 to review them; freeze (Step 9) refuses to run until `BENCHMARK_NAME` is set.

## 0 · Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: switch the runtime to GPU"

**Get the code.** For a private repo, add a GitHub token as a Colab secret named `GH_TOKEN` (key icon in the left sidebar, *Notebook access* on).

In [ ]:
REPO_URL = "https://github.com/SharviE29/cs670-proj.git"   # change if needed
BRANCH   = "main"
REPO_DIR = "/content/cs670-proj"

import os, subprocess
if not os.path.exists(REPO_DIR):
    url = REPO_URL
    try:
        from google.colab import userdata
        tok = userdata.get("GH_TOKEN")
        if tok:
            url = REPO_URL.replace("https://", f"https://{tok}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "-b", BRANCH, url, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
%cd {REPO_DIR}
!git log --oneline -1

**Install dependencies.** Colab's own CUDA build of torch is kept; everything else is pinned as in `requirements.txt` (Pillow/numpy/scipy versions affect rendered bytes).
If Colab asks you to **restart the session** after this cell, do it, then continue from the next cell (no need to re-run this one).

In [ ]:
!pip install -q transformers==5.20.2 accelerate==1.15.0 safetensors==0.8.0 tokenizers==0.23.3 \
    huggingface_hub==2.2.0 datasets==5.1.0 pyarrow==26.0.0 pillow==13.0.0 numpy==2.4.6 scipy==1.17.1 \
    pyyaml==6.0.3 pytest==9.2.0

**Settings.** Everything the pipeline needs is in a small override config written to `/content/config.colab.yaml` (outside the repo, so the git tree stays clean for `freeze`).

In [ ]:
import os, torch, textwrap
REPO_DIR   = "/content/cs670-proj"
DATA_ROOT  = "/content/cira_data"                 # fast local disk; checkpointed to Drive
FOOD101    = "/content/food-101"                  # official tarball layout (real Food-101 image IDs)
USE_DRIVE  = True
DRIVE_DIR  = "/content/drive/MyDrive/cira"        # checkpoints + final release
BENCHMARK_NAME = None                             # OPEN ITEM: set before Step 9, e.g. "CIRA-Food"

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
qwen_dtype = "float16" if "T4" in gpu else "bfloat16"   # T4 has no fast bf16
cfg_text = textwrap.dedent(f'''
    _base: {REPO_DIR}/data_pipeline/config.yaml
    benchmark_name: {BENCHMARK_NAME if BENCHMARK_NAME else "null"}
    paths:
      data_root: {DATA_ROOT}
      food101_root: {FOOD101}
      hf_cache: /content/hf_cache
    food101:
      source: local
    crops:
      device: cuda
      qwen_check:
        dtype: {qwen_dtype}
''')
open("/content/config.colab.yaml", "w").write(cfg_text)
os.environ["CIRA_CONFIG"] = "/content/config.colab.yaml"   # every `python -m data_pipeline...` uses it
os.environ["HF_HOME"] = "/content/hf_cache"
os.makedirs(DATA_ROOT, exist_ok=True)
%cd {REPO_DIR}
print(gpu, "| qwen dtype:", qwen_dtype); print(cfg_text)

**Drive checkpoints.** `save_checkpoint()` tars `DATA_ROOT` to Drive; the cell restores the newest checkpoint if `DATA_ROOT` is empty (e.g. after a disconnect).

In [ ]:
import glob, subprocess, time
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)

def save_checkpoint(tag):
    if not USE_DRIVE: return
    dst = f"{DRIVE_DIR}/ckpt_{tag}.tar"
    t = time.time()
    subprocess.run(["tar", "-cf", dst, "-C", os.path.dirname(DATA_ROOT), os.path.basename(DATA_ROOT)], check=True)
    print(f"checkpoint -> {dst} ({os.path.getsize(dst)/1e6:.0f} MB, {time.time()-t:.0f}s)")

def restore_latest():
    ck = sorted(glob.glob(f"{DRIVE_DIR}/ckpt_*.tar"), key=os.path.getmtime)
    if ck and not os.listdir(DATA_ROOT):
        print("restoring", ck[-1])
        subprocess.run(["tar", "-xf", ck[-1], "-C", os.path.dirname(DATA_ROOT)], check=True)
    else:
        print("nothing to restore" if not ck else "DATA_ROOT not empty; keeping it")

if USE_DRIVE: restore_latest()

**Sanity check:** class table + the test suite on a synthetic dataset (no GPU needed, ~1 min).

In [ ]:
!python -m data_pipeline.classes
!python -m pytest -q tests

## Step 2 · Food classes and dish cutouts
Downloads the official Food-101 tarball (~5 GB) so every cutout keeps its real Food-101 image ID. Only the **test** split is used.

In [ ]:
if not os.path.exists(f"{FOOD101}/meta/test.txt"):
    !wget -q --show-progress -O /content/food-101.tar.gz http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz
    !tar -xzf /content/food-101.tar.gz -C /content && rm /content/food-101.tar.gz
!wc -l {FOOD101}/meta/test.txt

**Extract cutouts** (GroundingDINO → SAM2 → filters → CLIP class check). Resumable: if the session dies, re-run Setup and this cell.
Add `--classes pho tacos` / `--limit_per_class 30` for a quick trial, `--fresh` to start over.

In [ ]:
!python -m data_pipeline.crops extract
save_checkpoint("crops")

**Qwen2.5-VL-3B crop-only check** (the one logged exception to the selection-bias rule), then apply the per-class rules.

In [ ]:
!python -m data_pipeline.crops qwen-check
!python -m data_pipeline.crops finalize
!python -m data_pipeline.crops report
save_checkpoint("crops_final")

In [ ]:
import json
from IPython.display import Image as IPImage, display
status = json.load(open(f"{DATA_ROOT}/crops/class_status.json"))
print("dropped classes:", {c: s["reasons"] for c, s in status.items() if not s["kept"]})
display(IPImage(f"{DATA_ROOT}/crops/contact_sheet.jpg", width=1400))

## Step 3 · Backgrounds (Wikimedia Commons)
`fetch` needs internet and respects Wikimedia's rate limit (1 request/s), so it is the slowest step. Per-image license, author and URL go to `backgrounds/licenses.csv`.
Try one region first: `--regions vietnam`.

In [ ]:
!python -m data_pipeline.backgrounds fetch
save_checkpoint("bg_fetch")
!python -m data_pipeline.backgrounds audit
!python -m data_pipeline.backgrounds report
save_checkpoint("bg_final")
display(IPImage(f"{DATA_ROOT}/backgrounds/contact_sheet.jpg", width=1400))

In [ ]:
import pandas as pd
lic = pd.read_csv(f"{DATA_ROOT}/backgrounds/licenses.csv")
print(lic["license"].value_counts().head(10)); lic.head()

## Steps 5–7 · Splits, groups, questions
Order: `splits assign` (who owns each photo/background) → `groups` → `questions` → `splits verify` (asserts no leakage).

In [ ]:
!python -m data_pipeline.splits assign
!python -m data_pipeline.groups
!python -m data_pipeline.questions
!python -m data_pipeline.splits verify
save_checkpoint("groups")

**One group's 21 cells** (7 images × 3 questions, with answers and relevance labels):

In [ ]:
first = json.loads(open(f"{DATA_ROOT}/groups/train.jsonl").readline())
print(json.dumps({k: first[k] for k in ["group_id","t","o","s","t_prime","o_prime","s_prime","regions","bg_dish_match"]}, indent=1))
!python -m data_pipeline.questions --show {first["group_id"]}

## Step 4 · Render
First one group side by side (keys x, t, o, b, to, tb, ob), then every split in parallel shards.

In [ ]:
!python -m data_pipeline.render --demo /content/group_demo.png
display(IPImage("/content/group_demo.png", width=1500))

In [ ]:
nproc = max(1, os.cpu_count())
for split in ["train", "val", "test_id", "test_unseen"]:
    procs = [subprocess.Popen(["python", "-m", "data_pipeline.render", "--split", split,
                               "--shard", str(i), "--num_shards", str(nproc)]) for i in range(nproc)]
    assert all(p.wait() == 0 for p in procs), f"render failed for {split}"
    print(split, "done")
!find {DATA_ROOT}/renders -name '*.png' | wc -l
save_checkpoint("renders")

## Step 8 · Controls and audits
All automated checks must pass before freezing: pixel preservation on **every** pair of **every** group, commutativity, labels, and saved PNG == fresh re-render.

In [ ]:
!python -m data_pipeline.audit controls
!python -m data_pipeline.audit checks --use_saved
!CIRA_TEST_FULL=1 python -m pytest -q tests
save_checkpoint("audited")

**Human audit.** Download `human_audit.html`, open it in a browser, mark each item, click *Download results CSV*, then upload that CSV in the next cell.

In [ ]:
!python -m data_pipeline.audit human-export
from google.colab import files
files.download(f"{DATA_ROOT}/audit/human_audit.html")

In [ ]:
up = files.upload()            # choose human_audit_results.csv
for name in up:
    !python -m data_pipeline.audit human-ingest "{name}"
save_checkpoint("human_audit")

## Step 9 · Freeze
Requires `BENCHMARK_NAME` (set it in the Settings cell and re-run that cell), passing checks, and a git commit (the clone provides it).
Writes `release/` with manifests, `FREEZE.json` (SHA-256 of every manifest and image + git commit + config), `STATS.md` and `DATASHEET.md`.

In [ ]:
!python -m data_pipeline.freeze
!cat {DATA_ROOT}/release/STATS.md | head -60

In [ ]:
if USE_DRIVE:
    !tar -czf {DRIVE_DIR}/release_and_renders.tgz -C {DATA_ROOT} release renders groups questions
    print("saved to", DRIVE_DIR)